# 강화학습 4강 — 인공지능 에이전트의 꿈 · OPTIMAL POLICY

2강에서 **탐험**했고, 3강에서 **기억**했습니다. 그렇게 배운 두뇌가 결국 손에 쥐는 것은 **정책(policy)** — 「모든 칸에서 이렇게 행동한다」는 화살표 지도입니다.
에이전트의 꿈은 이 지도를 가장 좋게 완성하는 것, **최적 정책(optimal policy)** 입니다.

그런데 오늘 호수에는 **코인**이 있습니다. 밟으면 +0.2, 잠시 뒤 다시 생깁니다. 별(G)은 +1.
두뇌는 지금 하는 게 최고인 줄 압니다. 알고 보면 더 큰 게 있었는데 — **코인 주변만 서성거리는 두뇌** 이야기입니다. (리워드 해킹 · 로컬 맥시마)

| 셀 | 하는 일 |
|---|---|
| 1 | 준비 |
| 2 | 코인 호수 열기 |
| 3 | 정책(policy) — 화살표 지도 |
| 4 | 최적 정책 π* — 가치 반복으로 정답 지도 구하기 |
| 5 | 할인율 γ — 별의 빛이 얼마나 멀리 닿나 (인내심) |
| 6 | 학습 함수 — 두뇌의 가치관(별 · 코인 · γ · 꿈)을 넣어 배우기 |
| 7 | 🪙 코인에 현혹된 두뇌 — 기본 가치관으로 500판 |
| 8 | 🎬 서성거리는 두뇌 영상 |
| 9 | 두뇌는 틀리지 않았다 — 두뇌의 계산 |
| 10 | ✨ 꿈을 크게 — 같은 호수, 다른 가치관 |
| 11 | ⚔️ 가치관 대결 8가지 (씨앗 5개) |
| 12 | 🏔️ 봉우리 지도 — 별 믿음 × 꿈 |
| 13 | ⏳ 할인율 실험 — 조급한 두뇌 · 너무 참는 두뇌 |
| 14 | 🚤 리워드 해킹 — 배 게임, 그리고 사람 |

> 위에서부터 순서대로 ▶ 실행하세요. GPU 는 필요 없습니다. (런타임 → 모두 실행 도 됩니다. 11·12번 셀은 각각 2~3분)

## 1. 준비

In [ ]:
!pip install -q "gymnasium[toy-text]"

import random
from collections import deque
import numpy as np
import gymnasium as gym
import matplotlib.pyplot as plt
from matplotlib import animation
from IPython.display import HTML, display

SEED = 0
학습률 = 0.5
plt.rcParams["figure.dpi"] = 110
화살표 = ["←", "↓", "→", "↑"]
print("준비 완료 · gymnasium", gym.__version__)

## 2. 코인 호수 열기

Gymnasium 의 얼음 호수(FrozenLake 8×8)에 **코인 3개**를 올립니다. 3강까지는 구멍(고통)만 있고 중간 보상이 없었죠.

- 코인을 밟으면 **+0.2**, 밟힌 코인은 **6걸음 뒤에 다시 생깁니다**
- 별(G)은 **+1** 이고 판이 끝납니다 · 구멍(H)은 0점으로 끝 · 한 판 최대 100걸음
- 코인 자리: 9 · 16 · 18 번 칸 (출발점 바로 아래쪽)

In [ ]:
class 코인호수:
    """FrozenLake 8x8 (미끄럽지 않음) + 코인 3개. 보상은 세상이 주는 '진짜' 보상."""
    코인자리 = [9, 16, 18]
    다시생김 = 6
    def __init__(self, render=False):
        self.env = gym.make("FrozenLake-v1", map_name="8x8", is_slippery=False, render_mode="rgb_array" if render else None)
        self.지도 = self.env.unwrapped.desc.astype(str)
        self.칸수 = self.env.observation_space.n
    def reset(self, seed=None):
        s, _ = self.env.reset(seed=seed)
        self.타이머 = [0, 0, 0]
        self.걸음 = 0
        return s
    def step(self, a):
        s2, r, terminated, truncated, _ = self.env.step(a)
        self.타이머 = [max(0, t - 1) for t in self.타이머]
        self.걸음 += 1
        코인 = False
        if not terminated and s2 in self.코인자리:
            k = self.코인자리.index(s2)
            if self.타이머[k] == 0:
                self.타이머[k] = self.다시생김
                코인 = True
        별 = terminated and r > 0
        끝 = terminated or truncated or self.걸음 >= 100
        return s2, dict(별=별, 코인=코인, 구멍=terminated and r == 0), 끝
    def render(self):
        return self.env.render()

호수 = 코인호수(render=True)
호수.reset(seed=SEED)
print("\n".join(" ".join(줄) for 줄 in 호수.지도))

def 호수_그리기(ax, 호수, 제목=""):
    ax.imshow(호수.render()); ax.axis("off"); ax.set_title(제목)
    for k, c in enumerate(호수.코인자리):
        y, x = divmod(c, 8)
        있음 = 호수.타이머[k] == 0
        ax.add_patch(plt.Circle((x * 64 + 32, y * 64 + 32), 16, color="#ffd447" if 있음 else "#444", ec="#7a5c00", lw=2))
fig, ax = plt.subplots(figsize=(4.4, 4.4)); 호수_그리기(ax, 호수, "coin lake · 3 coins near start"); plt.show()

# 무작위로 1000판 걸어 보면? 별에는 몇 번, 코인은 몇 개?
rng = random.Random(SEED)
별수, 코인수 = 0, 0
for i in range(1000):
    s = 호수.reset(); 끝 = False
    while not 끝:
        s, info, 끝 = 호수.step(rng.randrange(4))
        별수 += info["별"]; 코인수 += info["코인"]
print(f"무작위 1000판: 별 도착 {별수}번 · 코인 {코인수}개 (판당 {코인수/1000:.1f}개)")

## 3. 정책(policy) — 화살표 지도

**정책**은 「모든 칸에서 어떻게 행동할지」를 정해 둔 것입니다. 사람으로 치면 **가치관 · 습관**.
Q 표가 있으면 정책은 저절로 나옵니다 — 칸마다 **Q 가 가장 큰 화살표** 하나씩.

$$\pi(s) = \arg\max_a Q(s, a)$$

아직 아무것도 모르는 두뇌의 정책은 **무작위 화살표 지도**입니다. 되는대로 사는 삶이죠.

In [ ]:
def 정책_그리기(ax, Q, 호수, 제목="", 밝기=None):
    """칸마다 Q 가 가장 큰 화살표. 색 = 그 칸의 가장 큰 Q (밝을수록 좋다고 믿는 칸)"""
    V = Q.max(axis=1).reshape(8, 8)
    ax.imshow(V, cmap="viridis", vmin=0, vmax=밝기 if 밝기 else max(V.max(), 1e-9))
    for 칸 in range(64):
        y, x = divmod(칸, 8); 글자 = 호수.지도[y][x]
        if 글자 == "H":
            ax.add_patch(plt.Rectangle((x - .5, y - .5), 1, 1, color="#1b1b1b")); ax.text(x, y, "H", ha="center", va="center", color="#ff6b6b", weight="bold")
        elif 글자 == "G":
            ax.text(x, y, "★", ha="center", va="center", color="#ffd447", fontsize=16)
        else:
            if 칸 in 호수.코인자리: ax.add_patch(plt.Circle((x, y), .18, color="#ffd447", alpha=.9))
            ax.text(x, y, 화살표[int(Q[칸].argmax())], ha="center", va="center", color="white", fontsize=12)
        if 글자 == "S": ax.text(x - .42, y - .3, "S", color="white", fontsize=8, weight="bold")
    ax.set_xticks([]); ax.set_yticks([]); ax.set_title(제목, fontsize=11)

무작위_Q = np.random.default_rng(SEED).random((64, 4))     # 아무 숫자나 — 정책은 「가장 큰 것」만 보니 지도는 무작위
fig, ax = plt.subplots(figsize=(4.6, 4.6)); 정책_그리기(ax, 무작위_Q, 호수, "random policy — a life lived at random"); plt.show()

## 4. 최적 정책 π* — 정답 지도 구하기

코인이 **없는** 호수라면 정답 지도는 미리 계산할 수 있습니다. 세상의 규칙(어디로 가면 어디 도착하는지)을 다 아니까요.
칸마다 「이 행동을 하면 받을 보상 + γ × 다음 칸의 최고값」을 반복해서 채웁니다 — **가치 반복(value iteration)**.

$$Q^*(s,a) = r(s,a) + \gamma \max_{a'} Q^*(s',a') \qquad \pi^*(s) = \arg\max_a Q^*(s,a)$$

강화학습의 목적은 **이 지도(π\*)를 찾는 것**입니다. 다만 현실에선 세상의 규칙을 몰라서, 2·3강처럼 **걸어 보며** 배웁니다.

In [ ]:
def 다음칸(s, a):
    y, x = divmod(s, 8)
    if a == 0: x = max(0, x - 1)
    elif a == 1: y = min(7, y + 1)
    elif a == 2: x = min(7, x + 1)
    else: y = max(0, y - 1)
    return y * 8 + x

def 최적_Q(호수, 할인율=0.95, 별_믿음=1.0):
    """코인 없는 호수에서 가치 반복. 별 = 별_믿음, 구멍 = 0 (끝)"""
    Q = np.zeros((64, 4))
    for _ in range(300):
        새Q = Q.copy()
        for s in range(64):
            y, x = divmod(s, 8)
            if 호수.지도[y][x] in "HG": continue
            for a in range(4):
                s2 = 다음칸(s, a); y2, x2 = divmod(s2, 8); 글자 = 호수.지도[y2][x2]
                새Q[s, a] = 0.0 if 글자 == "H" else (별_믿음 if 글자 == "G" else 할인율 * Q[s2].max())
        if np.abs(새Q - Q).max() < 1e-9: break
        Q = 새Q
    return Q

Q별 = 최적_Q(호수)
fig, ax = plt.subplots(figsize=(4.6, 4.6)); 정책_그리기(ax, Q별, 호수, "optimal policy π* (no coins, γ=0.95)"); plt.show()
print(f"출발점의 최고 Q = {Q별[0].max():.3f}  (= 0.95^13 — 가장 짧은 길 14걸음)")

## 5. 할인율 γ — 별의 빛이 얼마나 멀리 닿나

γ(감마)는 **미래를 얼마나 쳐줄까**, 두뇌의 **인내심**입니다. 별의 1점은 한 칸 멀어질 때마다 γ 배로 작아집니다.
γ 가 작으면 출발점에서 별은 거의 보이지 않습니다 (0.5¹³ ≈ 0.0001). 눈앞만 보는 삶이죠.

In [ ]:
fig, axes = plt.subplots(1, 4, figsize=(16, 4.2))
for ax, g in zip(axes, [0.5, 0.8, 0.95, 0.99]):
    Qg = 최적_Q(호수, 할인율=g)
    정책_그리기(ax, Qg, 호수, f"γ = {g} · start value {Qg[0].max():.4f}", 밝기=1.0)
plt.show()
print("화살표(정책)는 같아도, 출발점에서 느끼는 별의 가치는 γ 에 따라 수만 배 다릅니다.")

## 6. 학습 함수 — 두뇌의 가치관을 넣어 배우기

세상이 주는 **진짜 보상**(별 +1, 코인 +0.2)은 바꿀 수 없습니다. 바꿀 수 있는 건 **두뇌가 믿는 보상 = 가치관** 입니다.

| 가치관 | 뜻 |
|---|---|
| `별_믿음` | 별을 얼마로 믿을까 (1 ~ 10) |
| `코인_믿음` | 코인을 얼마로 믿을까 (0 ~ 0.5) |
| `할인율` | 미래를 얼마나 쳐줄까 = 인내심 (0.5 ~ 0.99) |
| `꿈` | **가 보지 않은 길을 처음에 얼마로 믿을까** = Q 표의 시작값 (0 ~ 10). 크면 안 가 본 곳이 더 좋아 보여 끝까지 찾아 나선다 |

학습은 3강 방식 그대로 — Q 표 + 엡실론-그리디(1 → 0.01) + 기억 상자(1만 개, 걸음마다 16개 다시 배움).

In [ ]:
def 학습(별_믿음=1.0, 코인_믿음=0.2, 할인율=0.95, 꿈=0.0, 판수=500, seed=SEED, 사진_간격=10):
    rng = random.Random(seed)
    호수 = 코인호수()
    Q = np.zeros((64, 4))
    for s in range(64):                                   # 꿈: 구멍·별이 아닌 칸은 전부 「꿈」에서 시작
        y, x = divmod(s, 8)
        if 호수.지도[y][x] not in "HG": Q[s] = 꿈
    상자 = deque(maxlen=10000)
    엡실론, 줄이기 = 1.0, 0.01 ** (1 / (판수 * 0.6))
    기록 = dict(결과=[], 코인=[], 출발점_Q=[], 사진들=[])
    for 판 in range(판수):
        s = 호수.reset(); 끝 = False; 코인수 = 0; 결과 = "맴돎"
        while not 끝:
            if rng.random() < 엡실론: a = rng.randrange(4)
            else:
                최고 = Q[s].max(); a = rng.choice([i for i in range(4) if Q[s, i] == 최고])
            s2, info, 끝 = 호수.step(a)
            r = (별_믿음 if info["별"] else 0.0) + (코인_믿음 if info["코인"] else 0.0)   # 두뇌가 믿는 보상
            코인수 += info["코인"]
            if info["별"]: 결과 = "별"
            elif info["구멍"]: 결과 = "구멍"
            상자.append((s, a, r, s2, info["별"] or info["구멍"]))
            if len(상자) >= 16:
                for (s_, a_, r_, s2_, 끝_) in rng.sample(상자, 16):
                    목표값 = r_ if 끝_ else r_ + 할인율 * Q[s2_].max()
                    Q[s_, a_] += 학습률 * (목표값 - Q[s_, a_])
            s = s2
        기록["결과"].append(결과); 기록["코인"].append(코인수); 기록["출발점_Q"].append(Q[0].max())
        엡실론 = max(0.01, 엡실론 * 줄이기)
        if 판 % 사진_간격 == 0 or 판 == 판수 - 1: 기록["사진들"].append((판 + 1, Q.copy()))
    기록["Q"] = Q
    return 기록

def 다_배운_두뇌_시험(Q, 판수=20, seed=123):
    """탐험 없이(가장 큰 Q, 같으면 무작위) 20판: 별 도착 비율, 별에 도착한 판의 평균 코인"""
    rng = random.Random(seed); 호수 = 코인호수(); 별, 코인 = 0, 0
    for _ in range(판수):
        s = 호수.reset(); 끝 = False; c = 0
        while not 끝:
            최고 = Q[s].max(); a = rng.choice([i for i in range(4) if Q[s, i] == 최고])
            s, info, 끝 = 호수.step(a); c += info["코인"]
            if info["별"]: 별 += 1; 코인 += c
    return 별 / 판수, (코인 / 별 if 별 else 0.0)

def 요약(기록):
    n = len(기록["결과"]); 결과 = 기록["결과"]
    도착률, 코인_판 = 다_배운_두뇌_시험(기록["Q"])
    return dict(별=결과.count("별") / n, 맴돎=결과.count("맴돎") / n, 구멍=결과.count("구멍") / n,
                코인=np.mean(기록["코인"]), 다배운_도착=도착률, 다배운_코인=코인_판)
print("준비 완료")

## 7. 🪙 코인에 현혹된 두뇌 — 기본 가치관으로 500판

별 1 · 코인 0.2 · γ 0.95 · 꿈 0. 세상이 주는 진짜 보상을 그대로 믿는, 아주 「합리적인」 두뇌입니다.

In [ ]:
기본 = 학습()
요약_기본 = 요약(기본)
print(f"배우는 동안 500판: 별 도착 {요약_기본['별']:.1%} · 100걸음 내내 맴돎 {요약_기본['맴돎']:.1%} · 구멍 {요약_기본['구멍']:.1%} · 판당 코인 {요약_기본['코인']:.1f}개")
print(f"다 배운 두뇌(탐험 없이 20판): 별 도착 {요약_기본['다배운_도착']:.0%}")
fig, (a1, a2) = plt.subplots(1, 2, figsize=(10.5, 4.6))
정책_그리기(a1, 기본["Q"], 호수, "policy after 500 episodes (coins believed 0.2)")
a2.plot(기본["출발점_Q"], color="#ffd447", lw=2); a2.axhline(0.95 ** 13, ls="--", color="#8fd3ff", label="star path value 0.95^13 ≈ 0.51")
a2.set(xlabel="episode", ylabel="max Q at start"); a2.legend(); a2.grid(alpha=.3); a2.set_title("what the brain believes the start is worth")
plt.show()

## 8. 🎬 서성거리는 두뇌 영상

탐험 없이 걷게 하면 — 코인을 먹고, 다시 생길 때까지 옆 칸에서 기다렸다가 또 먹습니다. 그 주변만 서성거립니다.

In [ ]:
def 걸음_영상(Q, 최대=40, seed=7):
    호수 = 코인호수(render=True); rng = random.Random(seed)
    s = 호수.reset(); 장면들 = []; 끝 = False; 제목 = []
    코인수 = 0
    for t in range(최대):
        장면들.append((호수.render(), list(호수.타이머))); 제목.append(f"step {t} · coins {코인수}")
        최고 = Q[s].max(); a = rng.choice([i for i in range(4) if Q[s, i] == 최고])
        s, info, 끝 = 호수.step(a); 코인수 += info["코인"]
        if 끝:
            장면들.append((호수.render(), list(호수.타이머))); 제목.append("GOAL ★" if info["별"] else ("fell in a hole" if info["구멍"] else f"step {t+1} · coins {코인수}"))
            break
    return 장면들, 제목

장면들, 제목들 = 걸음_영상(기본["Q"])
fig, ax = plt.subplots(figsize=(4.6, 4.8))
그림 = ax.imshow(장면들[0][0]); ax.axis("off")
원들 = [plt.Circle(((c % 8) * 64 + 32, (c // 8) * 64 + 32), 16, color="#ffd447", ec="#7a5c00", lw=2) for c in 코인호수.코인자리]
for o in 원들: ax.add_patch(o)
def 장면(i):
    그림.set_data(장면들[i][0])
    for o, t in zip(원들, 장면들[i][1]): o.set_color("#ffd447" if t == 0 else "#444444")
    ax.set_title(제목들[i])
영상 = animation.FuncAnimation(fig, 장면, frames=len(장면들), interval=320)
plt.close(fig)
HTML(영상.to_jshtml())

## 9. 두뇌는 틀리지 않았다 — 두뇌의 계산

맴도는 두뇌를 비웃기 전에, 두뇌가 믿는 숫자로 계산해 봅시다.

- **별까지 가는 길**: 14걸음 뒤 1점 → 출발점에서 느끼는 가치는 0.95¹³ ≈ **0.51**
- **코인 사이를 왔다 갔다**: 6걸음마다 0.2 → 한 걸음당 0.033, 영원히 → 0.033 ÷ (1 − 0.95) ≈ **0.67** (게다가 Q 표는 「가장 좋았던 순간」을 기억해 이보다 더 크게 믿습니다)

**우리가 준 가치관대로라면 맴도는 게 진짜 최적입니다.** 두뇌가 틀린 게 아니라, 꿈(별)이 보상에 제대로 들어 있지 않았던 겁니다. 이게 **리워드 해킹**입니다.

In [ ]:
별길 = 0.95 ** 13
코인루프 = (0.2 / 6) / (1 - 0.95)
print(f"두뇌가 믿는 별 길의 가치       ≈ {별길:.2f}")
print(f"두뇌가 믿는 코인 루프의 가치    ≈ {코인루프:.2f}  (Q 표가 실제로 믿는 값: {기본['Q'][0].max():.2f})")
print("→ 코인 루프 > 별 길. 두뇌 입장에선 맴도는 게 '정답'.")
print()
for 별_믿음 in [1, 2, 4, 10]:
    print(f"별을 {별_믿음:2}점으로 믿으면 별 길의 가치 ≈ {별_믿음 * 별길:5.2f}  {'> 코인 루프 ✅' if 별_믿음 * 별길 > 기본['Q'][0].max() else '< 코인 루프 ❌'}")

## 10. ✨ 꿈을 크게 — 같은 호수, 다른 가치관

세상은 그대로 두고 **가치관만** 바꿉니다. 두 가지가 필요합니다.
1. **별을 코인보다 충분히 크게 믿기** (별_믿음 10) — 알고 나면 맴돌지 않는다
2. **꿈** (가 보지 않은 길을 3으로 믿기) — 코인에 만족하지 않고, 안 가 본 곳을 끝까지 찾아 나선다

In [ ]:
가치관들 = {
    "① 기본 (별1 · 꿈0)":          dict(별_믿음=1,  꿈=0),
    "② 별만 크게 (별10 · 꿈0)":     dict(별_믿음=10, 꿈=0),
    "③ 꿈만 크게 (별1 · 꿈3)":      dict(별_믿음=1,  꿈=3),
    "④ 둘 다 (별10 · 꿈3)":        dict(별_믿음=10, 꿈=3),
}
결과들 = {}
print(f"{'가치관':24} {'배우는 동안 별':>12} {'맴돎':>7} {'판당 코인':>9}   다 배운 두뇌")
for 이름, 설정 in 가치관들.items():
    기록 = 학습(**설정); 요 = 요약(기록); 결과들[이름] = (기록, 요)
    print(f"{이름:22} {요['별']:12.1%} {요['맴돎']:7.1%} {요['코인']:9.1f}   별 도착 {요['다배운_도착']:4.0%} · 코인 {요['다배운_코인']:.1f}개")

영문 = ["1 basic (star1 dream0)", "2 star10 dream0", "3 star1 dream3", "4 both (star10 dream3)"]
fig, axes = plt.subplots(1, 4, figsize=(17, 4.4))
for ax, 제목, (이름, (기록, 요)) in zip(axes, 영문, 결과들.items()):
    정책_그리기(ax, 기록["Q"], 호수, 제목 + f" · star {요['다배운_도착']:.0%}")
plt.show()

**읽는 법**
- **② 별만 크게**: 별이 10점인 걸 모릅니다. 코인에 만족한 뒤 탐험을 멈춰서 별을 **발견하지 못합니다.** 큰 봉우리가 있는 줄도 모르는 로컬 맥시마.
- **③ 꿈만 크게**: 별을 찾아냅니다. 그런데 별을 1점으로 믿으니 **코인 루프(0.67+)가 더 커 보여** 다시 코인으로 돌아갑니다.
- **④ 둘 다**: 찾아내고, 잊지 않습니다. 가는 길에 코인도 줍습니다 — 코인이 나쁜 게 아니라, 코인 때문에 별을 잊는 게 나쁜 겁니다.

## 11. ⚔️ 가치관 대결 8가지 (씨앗 5개)

운(씨앗)에 따라 결과가 흔들리니 **씨앗 5개 평균**으로 비교합니다. (2~3분)

In [ ]:
대결_설정 = {
    "1 basic (star1 dream0)":       dict(별_믿음=1,  코인_믿음=0.2, 꿈=0),
    "2 star10 dream0":              dict(별_믿음=10, 코인_믿음=0.2, 꿈=0),
    "3 star1 dream3":               dict(별_믿음=1,  코인_믿음=0.2, 꿈=3),
    "4 star4 dream3":               dict(별_믿음=4,  코인_믿음=0.2, 꿈=3),
    "5 star10 dream3":              dict(별_믿음=10, 코인_믿음=0.2, 꿈=3),
    "6 ignore coins (coin0 dream0)": dict(별_믿음=1,  코인_믿음=0.0, 꿈=0),
    "7 ignore coins + dream3":      dict(별_믿음=1,  코인_믿음=0.0, 꿈=3),
    "8 greedy coins (coin0.5 star10 dream3)": dict(별_믿음=10, 코인_믿음=0.5, 꿈=3),
}
씨앗들 = [0, 1, 2, 3, 4]
대결 = {}
for 이름, 설정 in 대결_설정.items():
    요들 = [요약(학습(seed=sd, 사진_간격=10_000, **설정)) for sd in 씨앗들]
    대결[이름] = dict(별=np.mean([q["다배운_도착"] for q in 요들]), 코인=np.mean([q["다배운_코인"] for q in 요들]), 맴돎=np.mean([q["맴돎"] for q in 요들]))
    print(f"{이름:40} 다 배운 두뇌 별 도착 {대결[이름]['별']:5.0%} · 별 판 코인 {대결[이름]['코인']:.2f}개 · 배우는 동안 맴돎 {대결[이름]['맴돎']:5.0%}")

이름들 = list(대결); y = np.arange(len(이름들))[::-1]
fig, ax = plt.subplots(figsize=(11, 4.6))
ax.barh(y, [대결[n]["별"] for n in 이름들], color=["#999", "#999", "#999", "#ffb000", "#2ca02c", "#8fd3ff", "#8fd3ff", "#2ca02c"])
for yi, n in zip(y, 이름들): ax.text(대결[n]["별"] + .01, yi, f"{대결[n]['별']:.0%} · coins {대결[n]['코인']:.1f}", va="center")
ax.set_yticks(y); ax.set_yticklabels(이름들); ax.set_xlim(0, 1.3); ax.set_xlabel("star arrival after learning (5 seeds avg)")
plt.show()

## 12. 🏔️ 봉우리 지도 — 별 믿음 × 꿈

가치관 두 개를 바둑판처럼 바꿔 보면 **지형**이 보입니다. 어두운 곳은 코인에 갇힌 작은 봉우리, 밝은 곳이 별까지 가는 큰 봉우리. (2~3분)

In [ ]:
별들 = [1, 2, 4, 7, 10]; 꿈들 = [0, 1, 3, 10]
지형 = np.zeros((len(꿈들), len(별들)))
for i, 꿈 in enumerate(꿈들):
    for j, 별 in enumerate(별들):
        지형[i, j] = np.mean([요약(학습(별_믿음=별, 꿈=꿈, seed=sd, 사진_간격=10_000))["다배운_도착"] for sd in [0, 1, 2]])
fig, ax = plt.subplots(figsize=(7, 4.6))
im = ax.imshow(지형, cmap="viridis", vmin=0, vmax=1, origin="lower")
ax.set_xticks(range(len(별들))); ax.set_xticklabels([f"star {b}" for b in 별들]); ax.set_yticks(range(len(꿈들))); ax.set_yticklabels([f"dream {d}" for d in 꿈들])
for i in range(len(꿈들)):
    for j in range(len(별들)): ax.text(j, i, f"{지형[i, j]:.0%}", ha="center", va="center", color="white" if 지형[i, j] < .6 else "black", weight="bold")
plt.colorbar(im, label="star arrival after learning"); ax.set_title("landscape: where the brain ends up")
plt.show()

## 13. ⏳ 할인율 실험 — 조급한 두뇌 · 너무 참는 두뇌

별 10 · 꿈 3 로 고정하고 γ 만 바꿉니다. 작으면 별이 너무 멀어 보이고(조급), 너무 크면 **영원히 이어지는 작은 코인 수입**이 한 번뿐인 별보다 커 보입니다.

In [ ]:
print(f"{'γ':>5} {'다 배운 두뇌 별 도착':>14} {'맴돎':>7}")
감마_결과 = {}
for g in [0.6, 0.8, 0.95, 0.99]:
    요들 = [요약(학습(별_믿음=10, 꿈=3, 할인율=g, seed=sd, 사진_간격=10_000)) for sd in [0, 1, 2]]
    감마_결과[g] = (np.mean([q["다배운_도착"] for q in 요들]), np.mean([q["맴돎"] for q in 요들]))
    print(f"{g:5} {감마_결과[g][0]:14.0%} {감마_결과[g][1]:7.0%}")
print("\n0.6: 10 × 0.6^13 ≈ 0.13 — 별이 너무 멀어 코인에 만족 · 0.99: 코인 수입 0.033/(1−0.99) ≈ 3.3 이 끝없이 이어져 별보다 커 보임")

## 14. 🚤 리워드 해킹 — 배 게임, 그리고 사람

2016년 OpenAI 가 공개한 실험이 있습니다. 보트 경주 게임(CoastRunners)에서 「점수를 최대로」 하라고 했더니, 에이전트는 **결승선으로 가지 않고** 중간에 다시 생기는 **보상 아이템이 있는 좁은 물길을 빙글빙글 돌며** 다른 배에 부딪히고 불이 붙으면서도 점수만 쌓았습니다. 사람 플레이어보다 점수가 20% 높았지만, 경주는 한 번도 끝내지 않았습니다.

오늘 우리 코인 두뇌와 똑같습니다. **보상이 꿈을 담지 못하면, 두뇌는 보상을 해킹합니다.**

사람도 그렇습니다. 월급 · 조회수 · 좋아요 · 눈앞의 칭찬이 코인입니다. 코인이 나쁜 게 아닙니다. 코인 때문에 **별이 있는 줄도 모르게 되는 것**, 그 주변만 서성거리게 되는 것이 문제입니다.

### 정리
1. **정책** = 모든 칸의 화살표 지도. 강화학습의 목적은 **최적 정책 π\*** 를 찾는 것.
2. **최적**은 보상 함수가 정한다. 코인 0.2 · 별 1 이면 맴도는 게 진짜 최적 — **리워드 해킹**.
3. **꿈(낙관)** 이 없으면 큰 봉우리를 발견하지 못하고, **별을 충분히 크게 믿지 않으면** 발견해도 돌아간다.
4. **γ** 는 인내심. 너무 조급해도, 너무 참아도 코인에 산다.
5. 코인은 주우면서 별까지 — 그게 가장 높은 점수였다.

🏆 **강화학습 컵 #3 · 에이전트의 꿈** — 가치관 다섯 숫자(별 · 코인 · γ · 꿈 · 판수)를 설계해 코인 호수 5개에서 겨룹니다 → https://www.aicitybuilders.com/rl-cup3

**다음 5강** — 칸이 64개라 Q 표로 충분했습니다. 화면 전체가 상태라면? 표 대신 **딥러닝**이 Q 를 계산합니다 — DQN.